# Assignment 2: Conversational AI System (Manual Semantic Search)

This notebook implements a chat-based AI system with three services:
1. API Calls (weather example)
2. Semantic Query (Sherlock Holmes ebook)
3. Function Calling (unit conversion)

Includes short-term memory, guardrails, and a Gradio interface.

In [ ]:
# --- Imports ---
import os
import requests
import gradio as gr

In [ ]:
# --- Service 1: API Calls (Weather) ---
API_KEY = 'YOUR_OPENWEATHERMAP_KEY'

def get_weather(city: str) -> str:
    url = f'http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric'
    response = requests.get(url).json()
    if response.get('cod') != 200:
        return f"Sorry, I couldn't find weather data for {city}."
    temp = response['main']['temp']
    description = response['weather'][0]['description']
    wind = response['wind']['speed']
    return f"The weather in {city} is {temp}°C with {description}. Wind speed is {wind} m/s."

In [ ]:
# --- Service 2: Semantic Query (Manual, Sherlock Holmes ebook) ---
TXT_PATH = './05_src/assignment_chat/data/sherlock.txt'

# Load and split the ebook into chunks
with open(TXT_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

CHUNK_SIZE = 500
chunks = [text[i:i+CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]

def semantic_query(query: str):
    # Naive search: return first chunk containing the query
    results = [c for c in chunks if query.lower() in c.lower()]
    return results[0] if results else "Sorry, I could not find any relevant info."

In [ ]:
# --- Service 3: Function Calling (Unit Conversion) ---
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    conversions = {
        'miles_km': 1.60934,
        'km_miles': 0.621371,
        'lbs_kg': 0.453592,
        'kg_lbs': 2.20462
    }
    key = f'{from_unit}_{to_unit}'
    if key not in conversions:
        return 'Sorry, I cannot convert those units.'
    converted = value * conversions[key]
    return f'{value} {from_unit} is approximately {converted:.2f} {to_unit}.'

In [ ]:
# --- Chat System with Guardrails & Memory ---
RESTRICTED_TOPICS = ['cats','dogs','horoscope','zodiac','taylor swift']
conversation_memory = []

def chat(user_input):
    if any(word.lower() in user_input.lower() for word in RESTRICTED_TOPICS):
        return 'Sorry, I cannot answer questions about that topic.'

    # Determine which service to use
    if 'weather' in user_input.lower():
        # Extract city after 'in'
        city = user_input.split('in')[-1].strip()
        response = get_weather(city)
    elif 'convert' in user_input.lower():
        parts = user_input.lower().split()
        try:
            value = float(parts[1])
            from_unit = parts[2]
            to_unit = parts[4]
            response = convert_units(value, from_unit, to_unit)
        except:
            response = "Please provide a valid conversion query like 'Convert 10 miles to km'."
    else:
        response = semantic_query(user_input)

    conversation_memory.append(('User', user_input))
    conversation_memory.append(('AI', response))
    return response

In [ ]:
# --- Gradio Chat Interface ---
with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    user_input = gr.Textbox(label='Your message')
    user_input.submit(chat, user_input, chatbot)

demo.launch()